# 10 · CI/CD: the SageMaker Pipeline, in a passing and a failing state
**AAI-540 · Group 4 · Criteo CTR**

Everything in notebooks 03–07 — process, train, evaluate, gate, register,
deploy — compiled into one parameterised DAG that reproduces the whole path
from data to a registered, scored model:

```
CriteoProcess ─> CriteoTrain ─> CriteoEvaluate ─> CriteoQualityGate
                                                       │
                           pass ──> CriteoCreateModel ─┼─> CriteoRegisterModel
                                                       │   └─> CriteoBatchTransform
                           fail ──> CriteoQualityGateFailed
```

**Two deliberate departures from the Module 6 lab:**

1. **The gate metric is log loss, not MSE.** The lab predicts abalone age — a
   regression target. Clicks are binary, so mean squared error is the wrong
   loss. Log loss is what the model optimises, what the design document commits
   to, and what the production monitors track; using anything else would leave
   the gate, the training objective and the monitors disagreeing about what
   "better" means.
2. **The gate is relative, not absolute.** `LogLossThreshold` is a pipeline
   parameter set to the incumbent's test log loss, so each execution has to
   beat the model already registered. The default is the benchmark from
   notebook 05 — the weakest defensible bar, because a model that cannot beat
   predicting the average click rate must never reach the registry.

The pipeline definition lives in `src/criteo_ctr/pipeline.py`. Its step and
parameter names, and the requirement that the gate have **both** branches, are
asserted in `tests/test_pipeline_contract.py` on every push — the one failure
mode you otherwise only find by paying for an execution.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import pandas as pd
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

from criteo_ctr import config as C
from criteo_ctr import pipeline as P
from criteo_ctr.io_utils import Store, s3_uri

sess = sagemaker.Session()
boto_sess = sess.boto_session
sm = boto_sess.client("sagemaker")
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)

ART_DIR = os.path.abspath("../artifacts"); os.makedirs(ART_DIR, exist_ok=True)

benchmark = json.loads(boto_sess.client("s3").get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/benchmark_metrics.json")["Body"].read())
gate_threshold = benchmark["gate_threshold_log_loss"]
print(f"region={region}\nbucket={bucket}")
print(f"gate threshold from notebook 05: {gate_threshold:.6f}")

## 1 · Compile the pipeline

A `PipelineSession` rather than a plain `Session` is what makes this work:
with it, `processor.run(...)` and `estimator.fit(...)` return *step arguments*
instead of launching jobs, so the whole graph is defined before anything
executes.

In [ ]:
pipeline_session = PipelineSession(boto_session=boto_sess, default_bucket=bucket)

pipeline = P.build_pipeline(
    session=pipeline_session, role=role, bucket=bucket, prefix=C.PREFIX,
    source_dir=os.path.abspath("../scripts"),
    base_log_loss=gate_threshold)

print(f"pipeline: {pipeline.name}")
print(f"steps   : {[s.name for s in pipeline.steps]}")
print("\nparameters:")
for spec in P.PARAMETER_SPEC:
    print(f"  {spec['name']:26s} {spec['type']:8s} default={spec['default']!r}")
    print(f"  {'':26s} {spec['purpose']}")

## 2 · Validate the definition before paying for it

`pipeline.validate_definition` checks the compiled JSON has every step, every
parameter, and a gate with **both** an `IfSteps` and an `ElseSteps` branch. A
gate missing its failure branch would end a bad execution as *successful* —
which is the single most expensive mistake available here, because it means an
under-performing model reaches the registry with a green tick.

In [ ]:
definition = json.loads(pipeline.definition())
check = P.validate_definition(definition)

print(f"definition valid: {check['passed']}")
for problem in check["problems"]:
    print(f"  PROBLEM: {problem}")
print(f"\nsteps in the graph ({len(check['all_steps'])}):")
for name in check["all_steps"]:
    print(f"  {name}")

with open(os.path.join(ART_DIR, "pipeline_definition.json"), "w") as fh:
    json.dump(definition, fh, indent=2)
store.put_json(definition, f"{C.ARTIFACTS_PREFIX}/pipeline_definition.json")
print(f"\nexported to artifacts/pipeline_definition.json — "
      f"tests/test_pipeline_contract.py re-validates it in CI")
assert check["passed"], check["problems"]

In [ ]:
pipeline.upsert(role_arn=role,
                description=("Criteo CTR: process -> train -> evaluate -> quality gate -> "
                             "create/register/transform, with a fail branch."))
print(f"upserted {pipeline.name}")
described = sm.describe_pipeline(PipelineName=pipeline.name)
print(f"status {described['PipelineStatus']}, "
      f"modified {described['LastModifiedTime']:%Y-%m-%d %H:%M}")

## 3 · Execution A — passing

Run at the default parameters, with the gate set to the notebook-05 benchmark.
The candidate should clear it comfortably, so the pass branch runs:
create model → register → batch transform.

In [ ]:
execution_pass = pipeline.start(
    parameters={"LogLossThreshold": gate_threshold,
                "ModelApprovalStatus": "PendingApproval"},
    execution_display_name="pass-vs-benchmark")
print(f"started {execution_pass.arn.split('/')[-1]}")

t0 = time.time()
execution_pass.wait(delay=30, max_attempts=80)     # ~15-20 minutes end to end
print(f"finished in {time.time() - t0:.0f}s: "
      f"{execution_pass.describe()['PipelineExecutionStatus']}")

In [ ]:
def step_table(execution):
    rows = []
    for step in execution.list_steps():
        meta = step.get("Metadata", {})
        kind = next(iter(meta), "")
        rows.append({
            "step": step["StepName"],
            "status": step["StepStatus"],
            "type": kind,
            "seconds": int((step["EndTime"] - step["StartTime"]).total_seconds())
                       if step.get("EndTime") and step.get("StartTime") else None,
            "detail": (meta.get(kind, {}).get("Arn", "")
                       or meta.get(kind, {}).get("Outcome", ""))[-48:],
        })
    return pd.DataFrame(rows)

steps_pass = step_table(execution_pass)
display(steps_pass)

print("Pass branch ran:", sorted(set(steps_pass["step"]) & set(P.ON_PASS_STEPS)))
print("Fail step ran  :", P.STEP_FAIL in set(steps_pass["step"]))

### The evaluation report the gate read

Downloaded from S3, exactly as the lab downloads `evaluation.json` — this is
the number the `ConditionStep` compared against `LogLossThreshold`.

In [ ]:
evaluation_uri = f"s3://{bucket}/{C.PREFIX}/pipeline/evaluation/evaluation.json"
local = os.path.join(ART_DIR, "pipeline_evaluation.json")
sagemaker.s3.S3Downloader.download(evaluation_uri, ART_DIR, sagemaker_session=sess)

evaluation = json.load(open(os.path.join(ART_DIR, "evaluation.json")))
metrics = evaluation["binary_classification_metrics"]
display(pd.DataFrame([{k: round(v["value"], 6) for k, v in metrics.items()
                       if isinstance(v, dict) and "value" in v}]).T.rename(
                           columns={0: "value"}))

candidate_log_loss = metrics["log_loss"]["value"]
print(f"\ngate read {P.GATE_METRIC_PATH} = {candidate_log_loss:.6f}")
print(f"threshold was              {gate_threshold:.6f}")
print(f"margin                     {gate_threshold - candidate_log_loss:.6f}")

In [ ]:
# The registry is the proof the pass branch completed: a new version, with the
# metrics above attached, waiting for a human.
versions = sm.list_model_packages(ModelPackageGroupName=P.MODEL_PACKAGE_GROUP,
                                  SortBy="CreationTime", SortOrder="Descending",
                                  MaxResults=10)["ModelPackageSummaryList"]
display(pd.DataFrame([{"version": v["ModelPackageVersion"],
                       "status": v["ModelApprovalStatus"],
                       "created": v["CreationTime"]} for v in versions]))

latest = sm.describe_model_package(ModelPackageName=versions[0]["ModelPackageArn"])
print(f"latest version {versions[0]['ModelPackageVersion']}: "
      f"{latest['ModelApprovalStatus']}")
print("metrics attached:",
      latest.get("ModelMetrics", {}).get("ModelQuality", {})
            .get("Statistics", {}).get("S3Uri", "none"))

## 4 · Execution B — failing, on purpose

Same code, same data, one parameter changed: the threshold is tightened below
what the model just scored. This is the demonstration that the gate actually
stops a model rather than merely existing — the lab's own point, made with a
threshold set to fail.

**Why this matters more than the passing run.** A pipeline that has only ever
succeeded has never shown that it would refuse anything.

In [ ]:
tightened = round(candidate_log_loss * 0.95, 6)   # 5% better than achieved: unreachable
print(f"tightening the gate to {tightened:.6f} "
      f"(the candidate scores {candidate_log_loss:.6f})")

execution_fail = pipeline.start(
    parameters={"LogLossThreshold": tightened},
    execution_display_name="fail-tightened-gate")
print(f"started {execution_fail.arn.split('/')[-1]}")

try:
    execution_fail.wait(delay=30, max_attempts=80)
except Exception as exc:
    # A FailStep makes the execution terminate as Failed; waiter raises. That is
    # the intended outcome here, not an error in the notebook.
    print(f"execution ended as expected: {type(exc).__name__}")

status = execution_fail.describe()["PipelineExecutionStatus"]
print(f"status: {status}")

In [ ]:
steps_fail = step_table(execution_fail)
display(steps_fail)

ran = set(steps_fail["step"])
print(f"fail step executed      : {P.STEP_FAIL in ran}")
print(f"register step suppressed: {P.STEP_REGISTER not in ran}")
print(f"transform suppressed    : {P.STEP_TRANSFORM not in ran}")

failure = execution_fail.describe().get("FailureReason", "")
print(f"\nfailure reason recorded on the execution:\n  {failure}")

gate_row = steps_fail[steps_fail["step"] == P.STEP_GATE]
if not gate_row.empty:
    print(f"\ngate outcome: {gate_row.iloc[0]['detail']}")

In [ ]:
# Side by side: the same graph, two outcomes. This table is what the video
# demonstration shows for the "CI/CD DAG in a successful and failed state"
# requirement.
comparison = (steps_pass[["step", "status"]].rename(columns={"status": "execution A (pass)"})
              .merge(steps_fail[["step", "status"]].rename(columns={"status": "execution B (fail)"}),
                     on="step", how="outer")
              .fillna("did not run"))
order = {name: i for i, name in enumerate(P.STEP_NAMES)}
comparison = comparison.sort_values("step", key=lambda s: s.map(order)).reset_index(drop=True)
display(comparison)

versions_after = sm.list_model_packages(
    ModelPackageGroupName=P.MODEL_PACKAGE_GROUP, MaxResults=10)["ModelPackageSummaryList"]
print(f"model package versions before: {len(versions)}, after the failed run: "
      f"{len(versions_after)}  ->  the failing candidate was not registered.")

## 5 · Lineage

Every artefact an execution touched — the preprocessing code, the datasets, the
model, the evaluation report. This is what makes a rollback possible: version 3
can be reinstated with the exact preprocessing script and data split it was
built from, rather than whatever the repository happens to contain today.

In [ ]:
from sagemaker.lineage.visualizer import LineageTableVisualizer

visualizer = LineageTableVisualizer(sagemaker.session.Session())
for step in execution_pass.list_steps():
    if step["StepStatus"] != "Succeeded":
        continue
    frame = visualizer.show(pipeline_execution_step=step)
    if frame is not None:
        print(f"\n=== {step['StepName']} ===")
        display(frame)

## 6 · What CI covers, and what it does not

`.github/workflows/ci.yml` runs on every push and pull request:

| Check | What it would catch |
|---|---|
| `pytest tests` (104 tests) | a broken transform, a leaking encoder, a metric computing the wrong thing, a monitor that no longer fires on degradation |
| `tests/test_pipeline_contract.py` | a dropped step, a renamed parameter, a gate with no failure branch |
| `tools/local_dry_run.py` | the whole path end to end on synthetic data, in about two seconds |
| notebook JSON parse | a corrupted notebook reaching `main` |

**What it does not do:** run a real pipeline execution. That needs AWS
credentials in the runner, which this environment does not provide, so the
compiled definition is validated offline and executions are started from this
notebook. That boundary is recorded in the design document rather than implied
away — "CI/CD" here means an automated, gated, reproducible path from data to a
registered model, not a GitHub runner with production credentials.

**The leakage test is the one worth naming.** `tests/test_datasets.py`
asserts the encoder's category counts equal counts computed from the training
split alone, and that a level first seen after the training window maps to the
rare bucket. Without it, nothing in the system would notice the single mistake
that would invalidate every number in the design document.

In [ ]:
record = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "pipeline_name": pipeline.name,
    "pipeline_arn": described["PipelineArn"],
    "definition_valid": check["passed"],
    "steps": check["all_steps"],
    "parameters": check["parameters"],
    "gate": {"metric_path": P.GATE_METRIC_PATH,
             "metric": "log_loss",
             "rationale": "binary target; MSE from the lab does not apply"},
    "execution_pass": {
        "arn": execution_pass.arn,
        "status": execution_pass.describe()["PipelineExecutionStatus"],
        "threshold": gate_threshold,
        "candidate_log_loss": candidate_log_loss,
        "steps": steps_pass.to_dict("records"),
    },
    "execution_fail": {
        "arn": execution_fail.arn,
        "status": status,
        "threshold": tightened,
        "candidate_log_loss": candidate_log_loss,
        "failure_reason": failure,
        "steps": steps_fail.to_dict("records"),
        "registered": False,
    },
    "model_package_versions": len(versions_after),
}
store.put_json(record, f"{C.ARTIFACTS_PREFIX}/cicd_record.json")
with open(os.path.join(ART_DIR, "cicd_record.json"), "w") as fh:
    json.dump(record, fh, indent=2, default=str)

print(json.dumps({k: v for k, v in record.items()
                  if k not in ("execution_pass", "execution_fail")}, indent=2,
                 default=str))
print("\nDeliverable checklist for the video is in docs/operation_validation.md")